# 2026 FIFA World Cup: Territorial Control vs Shot Creation Efficiency

## Purpose
This notebook analyzes whether territorial dominance translated into shot creation efficiency during the 2026 FIFA World Cup, using team-level metrics derived from FIFA official match statistics.

## Data
- File: `site_official_stats_team_wide_flagged.csv`
- Source: FIFA official Match Centre, rendered-site scrape
- Scope: all completed matches with full `FIFA Official Stats`
- Current structure after filtering: 103 matches, 206 team-match rows, 48 teams
- Excluded from the main calculation: Belgium vs Egypt (`match_id = 400021478`) because FIFA exposes only `Live Statistics` for that match, not the full `FIFA Official Stats` section.


## Public Storyline
1. Start with Field Tilt ranking to establish who controlled the most final-third territory.
2. Then use the Field Tilt vs Shot Creation Efficiency scatter plot to test the intuitive assumption that more territory means more efficient shot creation.
3. The main finding is that territory and shot creation efficiency were largely separate dimensions in this tournament.

## Derived Metrics

| Metric | Role | Formula | Aggregation |
|---|---|---|---|
| `field_tilt_proxy` | x-axis, territorial control | team final-third entries / (team + opponent final-third entries) * 100 | team tournament total |
| `shot_creation_efficiency` | y-axis, creation efficiency | attempts at goal / final-third entries * 100 | team tournament total |
| `goals_per_attempt` | auxiliary finishing metric | goals / attempts at goal * 100 | team tournament total |

## Aggregation Principle
Team-level metrics are calculated as `team tournament totals / team tournament totals`, not as the average of match-level ratios. This avoids giving the same weight to matches with very different event volumes.

## Visualization
- Scatter plot: x = Field Tilt Proxy, y = Shot Creation Efficiency
- Median reference lines split the plot into four interpretive zones
- Color = Goals per Attempt

## Publication Notes
- `Field Tilt Proxy` is a derived metric, not a directly published FIFA metric.
- Data source should be described as: FIFA official website match statistics.
- xG is not included because it is not available in the scraped FIFA official stats file.
- Opponent strength is not controlled in this version.

## Data Limitations for Publication
- Team match counts differ across the tournament because some teams played 3 matches and finalists played up to 8 matches. Spain-centered rankings are descriptive champion profiling, not a causal model of why Spain won.
- Belgium vs Egypt (`match_id = 400021478`) is excluded from the main analytical tables because FIFA provides only `Live Statistics` for that match. As a result, Belgium has 5 full-stat matches instead of 6, and Egypt has 4 full-stat matches instead of 5. Per-match metrics for those two teams can be slightly inflated because one real match is not in the denominator.





In [ ]:
"""
Step 1: Environment setup, data load, and flag-path mapping
===========================================================
Only BASE_DIR should need manual editing when this project folder is moved.
All other paths are built relative to BASE_DIR.

Expected project structure:
    Worldcup2026/
    ├── data/
    │   ├── fifa_worldcup_2026/site_scrape/site_official_stats_team_wide_flagged.csv
    │   └── flags/*.svg
    └── outputs/
"""

import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# Edit only this line if the project folder is moved
# =========================================================
BASE_DIR = Path.cwd().resolve()
for parent in [BASE_DIR, *BASE_DIR.parents]:
    if parent.name == "worldcup-2026-official-stats-analysis":
        BASE_DIR = parent
        break

DATA_DIR = BASE_DIR / "data" / "fifa_worldcup_2026" / "site_scrape"
RAW_CSV_PATH = DATA_DIR / "site_official_stats_team_wide_flagged.csv"
FLAGS_DIR = BASE_DIR / "data" / "flags"
OUTPUT_DIR = BASE_DIR / "outputs"

for p, name in [(BASE_DIR, "BASE_DIR"), (DATA_DIR, "DATA_DIR"), (FLAGS_DIR, "FLAGS_DIR")]:
    if not p.exists():
        raise FileNotFoundError(
            f"{name} path does not exist: {p}\n"
            f"If the project folder was moved, update BASE_DIR first."
        )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# Data load and source-quality filtering
# =========================================================
def load_raw_data(path: Path = RAW_CSV_PATH) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"CSV file not found: {path}")

    df = pd.read_csv(path)

    required_cols = {
        "match_id",
        "team_name",
        "team_side",
        "opponent_side",
        "stats_source",
        "stats_complete",
    }
    missing_required = required_cols - set(df.columns)
    if missing_required:
        raise ValueError(f"Missing required columns: {sorted(missing_required)}")

    print("[Raw data]")
    print(f"Rows: {len(df):,}")
    print(f"Matches: {df['match_id'].nunique():,}")
    print(f"Teams: {df['team_name'].nunique():,}")

    status_summary = (
        df[["match_id", "stats_source", "stats_complete"]]
        .drop_duplicates()
        .groupby(["stats_source", "stats_complete"])
        .size()
        .reset_index(name="matches")
    )
    display(status_summary)

    # Publication-safe filter: keep only matches where the FIFA site exposes
    # the full FIFA Official Stats section. Missing values are not imputed.
    df_complete = df[df["stats_complete"] == True].copy()

    excluded_matches = (
        df.loc[df["stats_complete"] != True, ["match_id", "match_url", "team_name", "stats_source", "parsed_stat_count"]]
        .sort_values(["match_id", "team_name"])
    )

    if not excluded_matches.empty:
        print("\n[Excluded incomplete matches]")
        display(excluded_matches)

    print("\n[Analysis data]")
    print(f"Rows: {len(df_complete):,}")
    print(f"Matches: {df_complete['match_id'].nunique():,}")
    print(f"Teams: {df_complete['team_name'].nunique():,}")

    return df_complete


df_raw = load_raw_data()


# =========================================================
# Flag file mapping
# =========================================================
TEAM_TO_FLAG_FILE = {
    "Algeria": "01_algeria.svg", "Argentina": "02_argentina.svg", "Australia": "03_australia.svg",
    "Austria": "04_austria.svg", "Belgium": "05_belgium.svg",
    "Bosnia and Herzegovina": "06_bosnia_and_herzegovina.svg", "Brazil": "07_brazil.svg",
    "Cabo Verde": "08_cabo_verde.svg", "Canada": "09_canada.svg", "Colombia": "10_colombia.svg",
    "Congo DR": "11_congo_dr.svg", "Croatia": "12_croatia.svg", "Curaçao": "13_cura_ao.svg",
    "Czechia": "14_czechia.svg", "Côte d'Ivoire": "15_c_te_d_ivoire.svg", "Ecuador": "16_ecuador.svg",
    "Egypt": "17_egypt.svg", "England": "18_england.svg", "France": "19_france.svg",
    "Germany": "20_germany.svg", "Ghana": "21_ghana.svg", "Haiti": "22_haiti.svg",
    "IR Iran": "23_ir_iran.svg", "Iraq": "24_iraq.svg", "Japan": "25_japan.svg",
    "Jordan": "26_jordan.svg", "Korea Republic": "27_korea_republic.svg", "Mexico": "28_mexico.svg",
    "Morocco": "29_morocco.svg", "Netherlands": "30_netherlands.svg", "New Zealand": "31_new_zealand.svg",
    "Norway": "32_norway.svg", "Panama": "33_panama.svg", "Paraguay": "34_paraguay.svg",
    "Portugal": "35_portugal.svg", "Qatar": "36_qatar.svg", "Saudi Arabia": "37_saudi_arabia.svg",
    "Scotland": "38_scotland.svg", "Senegal": "39_senegal.svg", "South Africa": "40_south_africa.svg",
    "Spain": "41_spain.svg", "Sweden": "42_sweden.svg", "Switzerland": "43_switzerland.svg",
    "Tunisia": "44_tunisia.svg", "Türkiye": "45_t_rkiye.svg", "USA": "46_usa.svg",
    "Uruguay": "47_uruguay.svg", "Uzbekistan": "48_uzbekistan.svg",
}

csv_teams = set(df_raw["team_name"].unique())
mapping_teams = set(TEAM_TO_FLAG_FILE.keys())
missing_in_mapping = csv_teams - mapping_teams
missing_in_csv = mapping_teams - csv_teams

print(f"\n[Flag mapping check] CSV teams: {len(csv_teams)}, mapped teams: {len(mapping_teams)}")
if missing_in_mapping or missing_in_csv:
    print(f"  [Warning] CSV only: {missing_in_mapping} / mapping only: {missing_in_csv}")
else:
    print("  [Check] Team names match the 48-team flag mapping.")

missing_files = [(t, f) for t, f in TEAM_TO_FLAG_FILE.items() if not (FLAGS_DIR / f).exists()]
if missing_files:
    print(f"\n[Warning] Missing flag files: {len(missing_files)}")
    display(pd.DataFrame(missing_files, columns=["team_name", "flag_file"]))
else:
    print("\n[Check] All 48 flag files exist.")




In [ ]:
"""
Step 2: Calculate core team-level metrics
=========================================
Input: df_raw from Step 1.

Aggregation principle:
- Team metrics are calculated from tournament totals.
- They are not averages of match-level ratios.
- Only complete FIFA Official Stats matches are used.
- No missing values are imputed.
"""

import pandas as pd
import numpy as np

# =========================================================
# Final-third entries total from five FIFA channel columns
# =========================================================
f3_cols = [c for c in df_raw.columns if "final_third_entries" in c]
expected_f3_count = 5

if len(f3_cols) != expected_f3_count:
    raise ValueError(f"Expected {expected_f3_count} final-third entry columns, found {len(f3_cols)}: {f3_cols}")

df_raw = df_raw.copy()
df_raw["f3"] = df_raw[f3_cols].sum(axis=1, min_count=expected_f3_count)

# =========================================================
# Add opponent final-third entries for Field Tilt Proxy
# =========================================================
opp_f3 = df_raw[["match_id", "team_side", "f3"]].rename(
    columns={"team_side": "opponent_side", "f3": "opp_f3"}
)

merged = df_raw.merge(
    opp_f3,
    on=["match_id", "opponent_side"],
    how="left",
    validate="many_to_one",
)

# =========================================================
# Metric 1: Field Tilt Proxy
# =========================================================
valid_tilt = merged.dropna(subset=["f3", "opp_f3"])

g_tilt = valid_tilt.groupby("team_name").agg(
    f3_sum=("f3", "sum"),
    opp_f3_sum=("opp_f3", "sum"),
    n_matches_tilt=("match_id", "nunique"),
)

g_tilt["field_tilt_proxy"] = (
    g_tilt["f3_sum"] / (g_tilt["f3_sum"] + g_tilt["opp_f3_sum"]) * 100
)

# Backward-compatible alias for existing plotting cells.
g_tilt["field_tilt"] = g_tilt["field_tilt_proxy"]

# =========================================================
# Metric 2: Shot Creation Efficiency
# =========================================================
valid_att = merged.dropna(subset=["f3", "attacking__attempts_at_goal__total"])

g_att = valid_att.groupby("team_name").agg(
    attempts_sum=("attacking__attempts_at_goal__total", "sum"),
    f3_sum2=("f3", "sum"),
)

g_att["shot_creation_efficiency"] = g_att["attempts_sum"] / g_att["f3_sum2"] * 100

# Backward-compatible alias for existing plotting cells.
g_att["attempts_per_f3"] = g_att["shot_creation_efficiency"]

# =========================================================
# Metric 3: Goals per Attempt
# =========================================================
g_goals = merged.groupby("team_name").agg(
    attempts_sum2=("attacking__attempts_at_goal__total", "sum"),
    goals_sum=("attacking__goal__total", "sum"),
    n_matches_total=("match_id", "nunique"),
)

g_goals["goals_per_attempt"] = g_goals["goals_sum"] / g_goals["attempts_sum2"] * 100

# =========================================================
# Final team-level table
# =========================================================
team_metrics = (
    g_tilt[["field_tilt_proxy", "field_tilt", "f3_sum", "opp_f3_sum", "n_matches_tilt"]]
    .join(g_att[["shot_creation_efficiency", "attempts_per_f3", "attempts_sum"]])
    .join(g_goals[["goals_per_attempt", "goals_sum", "n_matches_total"]])
)

team_metrics = team_metrics.sort_values("field_tilt_proxy", ascending=False)

print(f"[Check] Team metrics calculated: {len(team_metrics)} teams")
print(f"[Check] Match-count range: {team_metrics['n_matches_total'].min()} to {team_metrics['n_matches_total'].max()} matches")

# =========================================================
# Quadrant validation using median reference lines
# =========================================================
median_tilt = team_metrics["field_tilt_proxy"].median()
median_sce = team_metrics["shot_creation_efficiency"].median()

quadrants = {
    "High tilt / High SCE": team_metrics[(team_metrics.field_tilt_proxy >= median_tilt) & (team_metrics.shot_creation_efficiency >= median_sce)],
    "High tilt / Low SCE": team_metrics[(team_metrics.field_tilt_proxy >= median_tilt) & (team_metrics.shot_creation_efficiency < median_sce)],
    "Low tilt / Low SCE": team_metrics[(team_metrics.field_tilt_proxy < median_tilt) & (team_metrics.shot_creation_efficiency < median_sce)],
    "Low tilt / High SCE": team_metrics[(team_metrics.field_tilt_proxy < median_tilt) & (team_metrics.shot_creation_efficiency >= median_sce)],
}

print("\n[Quadrant check]")
for name, sub in quadrants.items():
    print(f"  {name}: {len(sub)} teams")

output_metrics = OUTPUT_DIR / "team_metrics_all_completed_matches.csv"
team_metrics.to_csv(output_metrics, encoding="utf-8-sig")
print(f"\nSaved team metrics: {output_metrics}")

display(team_metrics.round(2))




## Visualization 1: Field Tilt Ranking

This opening chart shows which teams controlled the largest share of final-third territory. It is designed to create the intuitive expectation that higher territory might lead to stronger attacking output. The next scatter plot tests that assumption directly.


In [ ]:
"""
Step 2A: Field Tilt Proxy ranking chart
=======================================
Purpose:
- Introduce the territorial-control ranking before the efficiency scatter plot.
- Show the top 10 and bottom 10 teams by Field Tilt Proxy.
- Set up the later contrast: high territory does not automatically mean high shot creation efficiency.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.close("all")

required_metrics = ["field_tilt_proxy", "shot_creation_efficiency", "n_matches_total"]
missing_metrics = [col for col in required_metrics if col not in team_metrics.columns]
if missing_metrics:
    raise ValueError(f"Missing required columns in team_metrics: {missing_metrics}")

ranking_df = (
    team_metrics
    .copy()
    .reset_index()
    .rename(columns={"index": "team_name"})
    .sort_values("field_tilt_proxy", ascending=False)
)
ranking_df["field_tilt_rank"] = np.arange(1, len(ranking_df) + 1)

plot_df = pd.concat([
    ranking_df.head(10),
    ranking_df.tail(10),
], ignore_index=True).drop_duplicates("team_name")

plot_df = plot_df.sort_values("field_tilt_proxy", ascending=True).copy()
plot_df["display_label"] = plot_df.apply(
    lambda row: f"#{int(row['field_tilt_rank'])}  {row['team_name']}",
    axis=1,
)

fig, ax = plt.subplots(figsize=(11.8, 8.0), dpi=150)
fig.patch.set_facecolor("#f8fafc")
ax.set_facecolor("#f8fafc")

bar_colors = []
for _, row in plot_df.iterrows():
    if row["team_name"] == "Spain":
        bar_colors.append("#dc2626")
    elif row["field_tilt_rank"] <= 10:
        bar_colors.append("#047857")
    else:
        bar_colors.append("#94a3b8")

bars = ax.barh(
    plot_df["display_label"],
    plot_df["field_tilt_proxy"],
    color=bar_colors,
    height=0.68,
    alpha=0.92,
)

for bar, (_, row) in zip(bars, plot_df.iterrows()):
    value = row["field_tilt_proxy"]
    ax.text(
        value + 0.8,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}% | {int(row['n_matches_total'])} matches",
        va="center",
        ha="left",
        fontsize=8.2,
        color="#111827",
        fontweight="bold" if row["team_name"] == "Spain" else "normal",
    )

ax.axvline(50, linestyle="--", color="#64748b", linewidth=1.0, alpha=0.80)
ax.text(50, len(plot_df) - 0.35, "Balanced territory", ha="center", va="bottom", fontsize=8.5, color="#64748b")

ax.set_xlim(0, max(80, plot_df["field_tilt_proxy"].max() + 14))
ax.set_xlabel("Field Tilt Proxy (%)", fontsize=11.0, color="#111827")
ax.set_title("Who Controlled the Most Final-Third Territory?", fontsize=16.0, pad=14, color="#111827")
ax.grid(axis="x", alpha=0.22)
ax.tick_params(axis="y", labelsize=9.2, colors="#111827")
ax.tick_params(axis="x", labelsize=9.2, colors="#475569")

for spine in ax.spines.values():
    spine.set_visible(False)

footnote = (
    "Top 10 and bottom 10 teams by Field Tilt Proxy. Spain is highlighted; labels show Field Tilt and matches played.\n"
    "Field Tilt Proxy = team final-third entries / both teams' final-third entries × 100.\n"
    "Team match counts differ from 3 to 8, so shorter samples may be more volatile.\n"
    "Source: FIFA Match Centre | Full Official Stats only."
)
fig.text(0.08, 0.025, footnote, fontsize=8.1, color="#64748b", linespacing=1.18)

plt.tight_layout(rect=[0, 0.075, 1, 1])

output_path = OUTPUT_DIR / "00_field_tilt_proxy_top_bottom_ranking_all_completed_matches.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

ranking_output = OUTPUT_DIR / "field_tilt_proxy_ranking_all_completed_matches.csv"
ranking_df.to_csv(ranking_output, index=False, encoding="utf-8-sig")

print(f"Saved figure: {output_path}")
print(f"Saved ranking CSV: {ranking_output}")
display(ranking_df[["field_tilt_rank", "team_name", "field_tilt_proxy", "shot_creation_efficiency", "n_matches_total"]].round(2))



In [ ]:
!pip install cairosvg


In [ ]:
"""
Step 3A: Field Tilt Proxy x Shot Creation Efficiency scatter plot with flags
============================================================================
Purpose:
- Create a flag-based scatter plot without heavy flag overlap.
- Keep each team's true data point visible as a faint dot.
- Move only the displayed flag positions when needed and draw a thin connector line.

Metric notes:
- field_tilt is the backward-compatible column for Field Tilt Proxy (%).
- attempts_per_f3 is the backward-compatible column for Shot Creation Efficiency (%).
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.image as mpimg
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from pathlib import Path
import io

try:
    import cairosvg
    CAIROSVG_AVAILABLE = True
except Exception as e:
    cairosvg = None
    CAIROSVG_AVAILABLE = False
    CAIROSVG_IMPORT_ERROR = e


# =========================================================
# Clean previous figures
# =========================================================
plt.close("all")

if not CAIROSVG_AVAILABLE:
    print("[Warning] Flag-image plot skipped because cairosvg/cairo is unavailable.")
    print(f"[Warning] Reason: {CAIROSVG_IMPORT_ERROR}")
else:

    # =========================================================
    # SVG to image array
    # =========================================================
    def load_flag_as_array(svg_path: Path, size: int = 64) -> np.ndarray:
        """Convert an SVG flag file into a pixel array readable by matplotlib."""
        png_bytes = cairosvg.svg2png(
            url=str(svg_path),
            output_width=size,
            output_height=size
        )
        return mpimg.imread(io.BytesIO(png_bytes), format="png")


    # =========================================================
    # Simple collision reduction for flag positions
    # =========================================================
    def spread_flag_positions(
        x_values,
        y_values,
        min_x_gap=0.055,
        min_y_gap=0.070,
        max_iter=900,
        step=0.010,
        padding=0.035,
    ):
        """
        Spread displayed flag positions in normalized 0-1 plot space.
        The original data coordinates are preserved separately.
        """
        x_values = np.asarray(x_values, dtype=float)
        y_values = np.asarray(y_values, dtype=float)

        x_min, x_max = x_values.min(), x_values.max()
        y_min, y_max = y_values.min(), y_values.max()
        x_range = max(x_max - x_min, 1e-9)
        y_range = max(y_max - y_min, 1e-9)

        pos = np.column_stack([
            (x_values - x_min) / x_range,
            (y_values - y_min) / y_range,
        ])

        original = pos.copy()
        n = len(pos)

        for _ in range(max_iter):
            moved = False

            for i in range(n):
                for j in range(i + 1, n):
                    dx = pos[j, 0] - pos[i, 0]
                    dy = pos[j, 1] - pos[i, 1]

                    if abs(dx) < min_x_gap and abs(dy) < min_y_gap:
                        moved = True

                        if abs(dx) < 1e-6:
                            dx = 1e-6 if original[j, 0] >= original[i, 0] else -1e-6
                        if abs(dy) < 1e-6:
                            dy = 1e-6 if original[j, 1] >= original[i, 1] else -1e-6

                        push_x = np.sign(dx) * (min_x_gap - abs(dx)) * step / min_x_gap
                        push_y = np.sign(dy) * (min_y_gap - abs(dy)) * step / min_y_gap

                        pos[i, 0] -= push_x
                        pos[j, 0] += push_x
                        pos[i, 1] -= push_y
                        pos[j, 1] += push_y

            # A light pull keeps moved flags close to their true data point.
            pos += (original - pos) * 0.002
            pos[:, 0] = np.clip(pos[:, 0], padding, 1 - padding)
            pos[:, 1] = np.clip(pos[:, 1], padding, 1 - padding)

            if not moved:
                break

        display_x = x_min + pos[:, 0] * x_range
        display_y = y_min + pos[:, 1] * y_range

        return display_x, display_y


    # =========================================================
    # Data preparation
    # =========================================================
    required_metrics = [
        "field_tilt",
        "attempts_per_f3",
        "goals_per_attempt"
    ]

    missing_metrics = [col for col in required_metrics if col not in team_metrics.columns]

    if missing_metrics:
        raise ValueError(
            f"Missing required columns in team_metrics: {missing_metrics}\n"
            f"Current columns: {team_metrics.columns.tolist()}"
        )

    plot_df = team_metrics.copy()
    plot_df["actual_x"] = plot_df["field_tilt"]
    plot_df["actual_y"] = plot_df["attempts_per_f3"]

    display_x, display_y = spread_flag_positions(
        plot_df["actual_x"].values,
        plot_df["actual_y"].values,
        min_x_gap=0.060,
        min_y_gap=0.075,
        max_iter=1200,
        step=0.012,
    )

    plot_df["display_x"] = display_x
    plot_df["display_y"] = display_y
    plot_df["flag_shift"] = np.sqrt(
        (plot_df["display_x"] - plot_df["actual_x"]) ** 2
        + (plot_df["display_y"] - plot_df["actual_y"]) ** 2
    )

    median_tilt = plot_df["field_tilt"].median()
    median_sce = plot_df["attempts_per_f3"].median()

    norm = mcolors.Normalize(
        vmin=plot_df["goals_per_attempt"].min(),
        vmax=plot_df["goals_per_attempt"].max()
    )

    cmap = plt.get_cmap("RdYlBu_r")


    # =========================================================
    # Figure setup
    # =========================================================
    fig = plt.figure(figsize=(14, 9), dpi=150)
    ax = fig.add_axes([0.08, 0.16, 0.74, 0.74])

    missing_flags = []

    x_min = min(plot_df["actual_x"].min(), plot_df["display_x"].min())
    x_max = max(plot_df["actual_x"].max(), plot_df["display_x"].max())
    y_min = min(plot_df["actual_y"].min(), plot_df["display_y"].min())
    y_max = max(plot_df["actual_y"].max(), plot_df["display_y"].max())

    ax.set_xlim(x_min - 4, x_max + 4)
    ax.set_ylim(y_min - 1.5, y_max + 1.5)


    # =========================================================
    # True data points and connector lines
    # =========================================================
    ax.scatter(
        plot_df["actual_x"],
        plot_df["actual_y"],
        s=18,
        color="#4b5563",
        alpha=0.35,
        zorder=1,
        label="True data point"
    )

    for team, row in plot_df.iterrows():
        if row["flag_shift"] > 0.25:
            ax.plot(
                [row["actual_x"], row["display_x"]],
                [row["actual_y"], row["display_y"]],
                color="#9ca3af",
                linewidth=0.7,
                alpha=0.65,
                zorder=2
            )


    # =========================================================
    # Plot flags at adjusted display positions
    # =========================================================
    for team, row in plot_df.iterrows():
        color = cmap(norm(row["goals_per_attempt"]))
        fname = TEAM_TO_FLAG_FILE.get(team)
        fpath = FLAGS_DIR / fname if fname else None

        if fpath is not None and fpath.exists():
            try:
                img_arr = load_flag_as_array(fpath, size=64)
                im = OffsetImage(img_arr, zoom=0.46)

                ab = AnnotationBbox(
                    im,
                    (row["display_x"], row["display_y"]),
                    frameon=True,
                    bboxprops=dict(
                        edgecolor=color,
                        linewidth=1.8,
                        boxstyle="round,pad=0.045"
                    ),
                    zorder=4
                )

                ax.add_artist(ab)

            except Exception as e:
                missing_flags.append((team, str(e)))
                ax.scatter(
                    row["display_x"],
                    row["display_y"],
                    color=color,
                    s=120,
                    edgecolors="black",
                    linewidth=0.6,
                    zorder=4
                )
                ax.annotate(
                    team,
                    (row["display_x"], row["display_y"]),
                    fontsize=6,
                    ha="center",
                    va="center",
                    zorder=5
                )
        else:
            missing_flags.append((team, "file not found"))
            ax.scatter(
                row["display_x"],
                row["display_y"],
                color=color,
                s=120,
                edgecolors="black",
                linewidth=0.6,
                zorder=4
            )
            ax.annotate(
                team,
                (row["display_x"], row["display_y"]),
                fontsize=6,
                ha="center",
                va="center",
                zorder=5
            )


    # =========================================================
    # Reference lines
    # =========================================================
    ax.axvline(
        median_tilt,
        linestyle="--",
        color="gray",
        linewidth=1,
        alpha=0.85,
        zorder=0
    )

    ax.axhline(
        median_sce,
        linestyle="--",
        color="gray",
        linewidth=1,
        alpha=0.85,
        zorder=0
    )

    ax.text(
        median_tilt,
        y_max + 1.0,
        "Median Field Tilt Proxy",
        fontsize=8,
        color="gray",
        ha="center",
        va="bottom"
    )

    ax.text(
        x_min - 3.5,
        median_sce,
        "Median SCE",
        fontsize=8,
        color="gray",
        ha="left",
        va="center"
    )


    # =========================================================
    # Labels and title
    # =========================================================
    ax.set_xlabel("Field Tilt Proxy (%)", fontsize=12)
    ax.set_ylabel("Shot Creation Efficiency (%)", fontsize=12)

    ax.set_title(
        "Territory Did Not Equal Shot Creation Efficiency",
        fontsize=16,
        pad=16
    )

    ax.grid(True, alpha=0.25)


    # =========================================================
    # Colorbar
    # =========================================================
    cax = fig.add_axes([0.86, 0.26, 0.024, 0.50])

    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Goals per Attempt (%)", fontsize=10)


    # =========================================================
    # Footnote
    # =========================================================
    footnote = (
        "Flags are shifted slightly only to reduce overlap; faint dots mark true data positions.\n"
        "Field Tilt Proxy = team final-third entries / both teams' final-third entries × 100.\n"
        "Shot Creation Efficiency = attempts at goal / final-third entries × 100.\n"
        "Belgium vs Egypt was excluded because FIFA only provides Live Statistics for that match.\n"
        "Data source: FIFA Match Centre | Sample: 103 completed matches with full FIFA Official Stats | "
        "xG not included; opponent strength not controlled."
    )

    fig.text(
        0.08,
        0.035,
        footnote,
        fontsize=8.2,
        color="gray"
    )


    # =========================================================
    # Save figure and adjusted-position table
    # =========================================================
    output_path = OUTPUT_DIR / "field_tilt_proxy_vs_shot_creation_efficiency_flags_adjusted_all_completed_matches.png"
    output_position_csv = OUTPUT_DIR / "team_metrics_field_tilt_flag_positions_all_completed_matches.csv"

    fig.savefig(
        output_path,
        dpi=220,
        bbox_inches="tight"
    )

    flag_position_cols = [
        "field_tilt",
        "attempts_per_f3",
        "goals_per_attempt",
        "actual_x",
        "actual_y",
        "display_x",
        "display_y",
        "flag_shift",
    ]

    plot_df[flag_position_cols].to_csv(
        output_position_csv,
        encoding="utf-8-sig"
    )

    plt.show()


    # =========================================================
    # Validation output
    # =========================================================
    print(f"[Check] Total teams: {len(plot_df)}")
    print(f"[Check] Flags rendered successfully: {len(plot_df) - len(missing_flags)}")
    print(f"[Check] Median Field Tilt Proxy: {median_tilt:.2f}%")
    print(f"[Check] Median Shot Creation Efficiency: {median_sce:.2f}%")
    print(f"[Check] Average flag shift: {plot_df['flag_shift'].mean():.2f}")
    print(f"[Check] Max flag shift: {plot_df['flag_shift'].max():.2f}")

    if missing_flags:
        print(f"[Warning] Failed to render flags for {len(missing_flags)} teams:")
        for team, err in missing_flags:
            print(f"  - {team}: {err}")
    else:
        print("[Check] All team flags rendered successfully.")

    print(f"Saved figure: {output_path}")
    print(f"Saved adjusted-position CSV: {output_position_csv}")





In [ ]:
"""
Prepare Field Tilt Proxy table by phase
=======================================

Run this before the Field Tilt ranking image code.
"""

import pandas as pd
import numpy as np

# =========================================================
# 1. Create competition phase
# =========================================================

GROUP_STAGE_PATH = DATA_DIR / "site_official_stats_team_wide_group_stage.csv"

group_stage_match_ids = set(
    pd.read_csv(GROUP_STAGE_PATH)["match_id"].unique()
)

df_field_tilt = df_raw.copy()

df_field_tilt["competition_phase"] = np.where(
    df_field_tilt["match_id"].isin(group_stage_match_ids),
    "Group Stage",
    "Knockout Stage"
)


# =========================================================
# 2. Calculate final-third entries total
# =========================================================

f3_cols = [c for c in df_field_tilt.columns if "final_third_entries" in c]

if len(f3_cols) != 5:
    raise ValueError(
        f"Expected 5 final-third entry columns, found {len(f3_cols)}: {f3_cols}"
    )

df_field_tilt["final_third_entries_total"] = (
    df_field_tilt[f3_cols].sum(axis=1, min_count=5)
)


# =========================================================
# 3. Add opponent final-third entries
# =========================================================

opponent_f3 = (
    df_field_tilt[
        [
            "match_id",
            "team_side",
            "final_third_entries_total"
        ]
    ]
    .rename(
        columns={
            "team_side": "opponent_side",
            "final_third_entries_total": "opponent_final_third_entries_total"
        }
    )
)

df_field_tilt = df_field_tilt.merge(
    opponent_f3,
    on=["match_id", "opponent_side"],
    how="left",
    validate="many_to_one"
)


# =========================================================
# 4. Function: aggregate Field Tilt Proxy by team
# =========================================================

def aggregate_field_tilt(input_df, phase_label):
    temp = input_df.dropna(
        subset=[
            "final_third_entries_total",
            "opponent_final_third_entries_total"
        ]
    ).copy()

    result = (
        temp
        .groupby("team_name")
        .agg(
            final_third_entries_total=("final_third_entries_total", "sum"),
            opponent_final_third_entries_total=("opponent_final_third_entries_total", "sum"),
            matches_used=("match_id", "nunique")
        )
    )

    result["field_tilt_proxy"] = (
        result["final_third_entries_total"]
        / (
            result["final_third_entries_total"]
            + result["opponent_final_third_entries_total"]
        )
        * 100
    )

    result["competition_phase"] = phase_label

    return result.reset_index()


# =========================================================
# 5. Create Field Tilt tables
# =========================================================

field_tilt_overall = aggregate_field_tilt(
    df_field_tilt,
    "Overall"
)

field_tilt_group_stage = aggregate_field_tilt(
    df_field_tilt[df_field_tilt["competition_phase"] == "Group Stage"],
    "Group Stage"
)

field_tilt_knockout_stage = aggregate_field_tilt(
    df_field_tilt[df_field_tilt["competition_phase"] == "Knockout Stage"],
    "Knockout Stage"
)

field_tilt_by_phase = pd.concat(
    [
        field_tilt_overall,
        field_tilt_group_stage,
        field_tilt_knockout_stage
    ],
    ignore_index=True
)


# =========================================================
# 6. Save result
# =========================================================

field_tilt_team_path = OUTPUT_DIR / "field_tilt_proxy_team_by_phase.csv"

field_tilt_by_phase.to_csv(
    field_tilt_team_path,
    index=False,
    encoding="utf-8-sig"
)

print("[Check] field_tilt_by_phase created")
print(f"Rows: {len(field_tilt_by_phase)}")
print(f"Saved: {field_tilt_team_path}")

display(
    field_tilt_by_phase
    .sort_values(["competition_phase", "field_tilt_proxy"], ascending=[True, False])
    .round(2)
)

In [ ]:
!pip install adjustText


In [ ]:
 """
Step 3B: Field Tilt Proxy x Shot Creation Efficiency scatter plot (final version v2)
==================================================================================
Revision notes:
- Fix 1: Avoid overlap between the Median SCE label and reference line.
- Fix 2: Make the Colombia/Netherlands finishing contrast explicit.
- Fix 3: Move the highlight box into open space and connect it with an arrow.
- Fix 4: Scale point size by matches played to show sample-size stability.

Required: team_metrics, OUTPUT_DIR, adjustText
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from adjustText import adjust_text
from scipy import stats

plt.close("all")

required_metrics = ["field_tilt", "attempts_per_f3", "goals_per_attempt", "n_matches_total"]
missing_metrics = [c for c in required_metrics if c not in team_metrics.columns]
if missing_metrics:
    raise ValueError(f"Missing required columns in team_metrics: {missing_metrics}")
if team_metrics.empty:
    raise ValueError("team_metrics is empty.")

DISPLAY_NAME_MAP = {
    "Curaçao": "Curacao",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Türkiye": "Turkiye",
    "IR Iran": "Iran",
}

plot_df = team_metrics.copy()
plot_df["display_team_name"] = [DISPLAY_NAME_MAP.get(t, t) for t in plot_df.index]

median_tilt = plot_df["field_tilt"].median()
median_sce = plot_df["attempts_per_f3"].median()

norm = mcolors.Normalize(
    vmin=plot_df["goals_per_attempt"].min(),
    vmax=plot_df["goals_per_attempt"].max()
)
cmap = plt.get_cmap("RdYlBu_r")

min_matches = plot_df["n_matches_total"].min()
max_matches = plot_df["n_matches_total"].max()
size_min = 38
size_max = 145

if min_matches == max_matches:
    plot_df["point_size"] = (size_min + size_max) / 2
else:
    plot_df["point_size"] = (
        size_min
        + (plot_df["n_matches_total"] - min_matches)
        / (max_matches - min_matches)
        * (size_max - size_min)
    )

trend_result = stats.linregress(plot_df["field_tilt"], plot_df["attempts_per_f3"])
trend_x = np.linspace(plot_df["field_tilt"].min(), plot_df["field_tilt"].max(), 200)
trend_y = trend_result.intercept + trend_result.slope * trend_x
trend_r2 = trend_result.rvalue ** 2
trend_p = trend_result.pvalue

focus_teams = ["Colombia", "Netherlands"]
missing_focus = [t for t in focus_teams if t not in plot_df.index]
if missing_focus:
    raise ValueError(f"Focus teams not found in the dataset: {missing_focus}")

focus_gap_ratio = plot_df.loc["Colombia", "attempts_per_f3"] / plot_df.loc["Netherlands", "attempts_per_f3"]
colombia_gpa = plot_df.loc["Colombia", "goals_per_attempt"]
netherlands_gpa = plot_df.loc["Netherlands", "goals_per_attempt"]

fig = plt.figure(figsize=(15, 10), dpi=150)
ax = fig.add_axes([0.07, 0.16, 0.74, 0.72])

ax.plot(
    trend_x,
    trend_y,
    linestyle=(0, (4, 4)),
    color="#9ca3af",
    linewidth=1.5,
    alpha=0.55,
    zorder=1,
)

ax.text(
    0.985,
    0.965,
    f"Linear trend: slope={trend_result.slope:.3f}, R²={trend_r2:.1%}, p={trend_p:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=8.0,
    color="#6b7280",
)

texts = []
for team, row in plot_df.iterrows():
    x, y = row["field_tilt"], row["attempts_per_f3"]
    color = cmap(norm(row["goals_per_attempt"]))
    is_focus = team in focus_teams

    point_size = row["point_size"] * (1.35 if is_focus else 1.0)

    ax.scatter(
        x,
        y,
        color=color,
        s=point_size,
        edgecolors="#111827" if is_focus else "white",
        linewidth=1.3 if is_focus else 0.6,
        alpha=1.0 if is_focus else 0.78,
        zorder=8 if is_focus else 3,
    )

    text = ax.text(
        x,
        y,
        row["display_team_name"],
        fontsize=8.0 if is_focus else 7,
        fontweight="bold" if is_focus else "medium",
        ha="center",
        va="center",
        color="#111827" if is_focus else "black",
        alpha=1.0 if is_focus else 0.78,
        zorder=9 if is_focus else 4,
    )
    texts.append(text)

ax.axvline(median_tilt, linestyle="--", color="gray", linewidth=1, alpha=0.85, zorder=1)
ax.axhline(median_sce, linestyle="--", color="gray", linewidth=1, alpha=0.85, zorder=1)

ax.text(
    median_tilt,
    plot_df["attempts_per_f3"].max() + 1.2,
    "Median Field Tilt Proxy",
    fontsize=8,
    color="gray",
    ha="center",
    va="bottom",
)

ax.text(
    plot_df["field_tilt"].min() - 4.5,
    median_sce + 0.55,
    "Median SCE",
    fontsize=8,
    color="gray",
    ha="left",
    va="bottom",
    bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85),
)

x_min, x_max = plot_df["field_tilt"].min(), plot_df["field_tilt"].max()
y_min, y_max = plot_df["attempts_per_f3"].min(), plot_df["attempts_per_f3"].max()
ax.set_xlim(x_min - 5, x_max + 5)
ax.set_ylim(y_min - 2, y_max + 2)

adjust_text(
    texts,
    ax=ax,
    expand_text=(1.25, 1.35),
    expand_points=(1.25, 1.35),
    force_text=(0.35, 0.55),
    force_points=(0.25, 0.45),
    lim=500,
    arrowprops=dict(arrowstyle="-", color="#cbd5e1", lw=0.45, alpha=0.45),
)

colombia = plot_df.loc["Colombia"]
netherlands = plot_df.loc["Netherlands"]
x_bracket = max(colombia["field_tilt"], netherlands["field_tilt"]) + 2.0
y_low = min(colombia["attempts_per_f3"], netherlands["attempts_per_f3"])
y_high = max(colombia["attempts_per_f3"], netherlands["attempts_per_f3"])
y_mid = (y_low + y_high) / 2

ax.plot([x_bracket, x_bracket], [y_low, y_high], color="#111827", linewidth=1.6, zorder=10)
ax.plot([x_bracket - 0.85, x_bracket], [y_high, y_high], color="#111827", linewidth=1.6, zorder=10)
ax.plot([x_bracket - 0.85, x_bracket], [y_low, y_low], color="#111827", linewidth=1.6, zorder=10)

for _, row in pd.DataFrame([colombia, netherlands]).iterrows():
    ax.plot(
        [row["field_tilt"], x_bracket - 0.85],
        [row["attempts_per_f3"], row["attempts_per_f3"]],
        color="#111827",
        linewidth=0.9,
        alpha=0.75,
        zorder=9,
    )

# Move the annotation box into open space and connect it to the bracket.
box_x, box_y = x_bracket + 2.0, 30.5
ax.annotate(
    f"Similar Field Tilt,\n{focus_gap_ratio:.1f}x SCE gap\n"
    f"(Goals/Attempt: Colombia {colombia_gpa:.0f}% vs Netherlands {netherlands_gpa:.0f}%)",
    xy=(x_bracket, y_mid),
    xycoords="data",
    xytext=(box_x, box_y),
    textcoords="data",
    ha="left",
    va="center",
    fontsize=8.6,
    color="#111827",
    fontweight="bold",
    bbox=dict(
        boxstyle="round,pad=0.32",
        facecolor="#f8fafc",
        edgecolor="#111827",
        linewidth=0.8,
        alpha=0.94,
    ),
    arrowprops=dict(
        arrowstyle="-",
        color="#111827",
        lw=1.0,
        alpha=0.6,
        connectionstyle="arc3,rad=0.15",
    ),
    zorder=11,
)

ax.set_xlabel("Field Tilt Proxy (%)", fontsize=13)
ax.set_ylabel("Shot Creation Efficiency (%)", fontsize=13)
ax.set_title("Territory Did Not Guarantee Shot Creation", fontsize=16, pad=18)
ax.grid(True, alpha=0.25)

cax = fig.add_axes([0.85, 0.25, 0.022, 0.50])
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Goals per Attempt (%)", fontsize=11)

# Point-size legend: teams with more matches have more stable tournament averages.
legend_match_counts = [int(min_matches), 5, int(max_matches)]
legend_match_counts = sorted(set([m for m in legend_match_counts if min_matches <= m <= max_matches]))

legend_handles = []
for match_count in legend_match_counts:
    if min_matches == max_matches:
        legend_size = (size_min + size_max) / 2
    else:
        legend_size = size_min + (match_count - min_matches) / (max_matches - min_matches) * (size_max - size_min)

    legend_handles.append(
        ax.scatter(
            [],
            [],
            s=legend_size,
            color="#9ca3af",
            edgecolors="#111827",
            linewidth=0.6,
            alpha=0.65,
            label=f"{match_count} matches",
        )
    )

size_legend = ax.legend(
    handles=legend_handles,
    title="Matches played",
    loc="lower right",
    frameon=True,
    framealpha=0.92,
    facecolor="#f8fafc",
    edgecolor="#cbd5e1",
    fontsize=8,
    title_fontsize=8.5,
)
ax.add_artist(size_legend)

footnote = (
    "Trend line is contextual only: slope=-0.011, R²=0.1%, p=0.834; point size reflects matches played.\n"
    "Teams with fewer matches may have more volatile tournament averages.\n"
    "Field Tilt Proxy = team final-third entries / both teams' final-third entries × 100; "
    "SCE = attempts at goal / final-third entries × 100.\n"
    "Colombia and Netherlands had near-identical Field Tilt but opposite profiles: "
    "Colombia created more shots per entry but converted fewer;\n"
    "Netherlands created fewer shots but converted more.\n"
    "Source: FIFA Match Centre | Full Official Stats only | xG and opponent strength not controlled."
)
fig.text(0.07, 0.035, footnote, fontsize=8.0, color="gray")

output_path = OUTPUT_DIR / "field_tilt_proxy_vs_shot_creation_efficiency_all_completed_matches.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight")
plt.show()

team_metrics_tableau = plot_df.copy().reset_index().rename(columns={"index": "team_name"})
output_csv = OUTPUT_DIR / "team_metrics_field_tilt_tableau_all_completed_matches.csv"
team_metrics_tableau.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"[Check] Total teams plotted: {len(plot_df)}")
print(f"[Check] Total labels: {len(texts)}")
print(f"[Check] Median Field Tilt Proxy: {median_tilt:.2f}%")
print(f"[Check] Median Shot Creation Efficiency: {median_sce:.2f}%")
print(f"[Check] Match count range: {int(min_matches)}-{int(max_matches)}")
print(f"[Check] Colombia matches: {int(colombia['n_matches_total'])} / Netherlands matches: {int(netherlands['n_matches_total'])}")
print(f"[Check] Colombia Goals/Attempt: {colombia_gpa:.1f}% / Netherlands: {netherlands_gpa:.1f}%")
print(f"Saved figure: {output_path}")
print(f"Saved Tableau CSV: {output_csv}")

In [ ]:
 """
Step 3B: Field Tilt Proxy x Shot Creation Efficiency scatter plot (final version v5)
====================================================================
Changes from v4:
- Point size keeps n_matches_total.
- Brackets, arrows, and text box removed. The style is aligned with notebooks 03 and 05: bold labels, larger markers, and darker edges.
- Colombia and Netherlands are highlighted. Specific values (5% and 24%) are retained in the footnote.
- Color still represents goals_per_attempt, while point size represents match count to avoid overlapping encodings.

Required: team_metrics(field_tilt, attempts_per_f3, goals_per_attempt, n_matches_total), OUTPUT_DIR, adjustText
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from adjustText import adjust_text

plt.close("all")

required_metrics = ["field_tilt", "attempts_per_f3", "goals_per_attempt", "n_matches_total"]
missing_metrics = [c for c in required_metrics if c not in team_metrics.columns]
if missing_metrics:
    raise ValueError(f"Missing required columns in team_metrics: {missing_metrics}")
if team_metrics.empty:
    raise ValueError("team_metrics is empty")

DISPLAY_NAME_MAP = {
    "Curaçao": "Curacao",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Türkiye": "Turkiye",
    "IR Iran": "Iran",
}

plot_df = team_metrics.copy()
plot_df["display_team_name"] = [DISPLAY_NAME_MAP.get(t, t) for t in plot_df.index]

median_tilt = plot_df["field_tilt"].median()
median_sce = plot_df["attempts_per_f3"].median()

norm = mcolors.Normalize(
    vmin=plot_df["goals_per_attempt"].min(),
    vmax=plot_df["goals_per_attempt"].max()
)
cmap = plt.get_cmap("RdYlBu_r")

min_matches = plot_df["n_matches_total"].min()
max_matches = plot_df["n_matches_total"].max()
size_min, size_max = 38, 145
if min_matches == max_matches:
    plot_df["point_size"] = (size_min + size_max) / 2
else:
    plot_df["point_size"] = size_min + (plot_df["n_matches_total"] - min_matches) / (max_matches - min_matches) * (size_max - size_min)

focus_teams = ["Colombia", "Netherlands"]
missing_focus = [t for t in focus_teams if t not in plot_df.index]
if missing_focus:
    raise ValueError(f"Focus teams not found in data: {missing_focus}")

colombia_gpa = plot_df.loc["Colombia", "goals_per_attempt"]
netherlands_gpa = plot_df.loc["Netherlands", "goals_per_attempt"]

fig = plt.figure(figsize=(15, 10), dpi=150)
ax = fig.add_axes([0.07, 0.16, 0.74, 0.72])

texts = []
for team, row in plot_df.iterrows():
    x, y = row["field_tilt"], row["attempts_per_f3"]
    color = cmap(norm(row["goals_per_attempt"]))
    is_focus = team in focus_teams

    ax.scatter(x, y, color=color, s=row["point_size"] * (1.2 if is_focus else 1.0),
               edgecolors="#111827" if is_focus else "white",
               linewidth=1.4 if is_focus else 0.6,
               alpha=1.0 if is_focus else 0.78,
               zorder=8 if is_focus else 3)

    t = ax.text(x, y, row["display_team_name"],
                 fontsize=8.2 if is_focus else 7,
                 fontweight="bold" if is_focus else "medium",
                 ha="center", va="center",
                 color="#111827" if is_focus else "black",
                 alpha=1.0 if is_focus else 0.78,
                 zorder=9 if is_focus else 4)
    texts.append(t)

ax.axvline(median_tilt, linestyle="--", color="gray", linewidth=1, alpha=0.85, zorder=1)
ax.axhline(median_sce, linestyle="--", color="gray", linewidth=1, alpha=0.85, zorder=1)

ax.text(median_tilt, plot_df["attempts_per_f3"].max() + 1.2, "Median Field Tilt Proxy",
        fontsize=8, color="gray", ha="center", va="bottom")
ax.text(plot_df["field_tilt"].min() - 4.5, median_sce + 0.55, "Median SCE",
        fontsize=8, color="gray", ha="left", va="bottom",
        bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85))

x_min, x_max = plot_df["field_tilt"].min(), plot_df["field_tilt"].max()
y_min, y_max = plot_df["attempts_per_f3"].min(), plot_df["attempts_per_f3"].max()
ax.set_xlim(x_min - 5, x_max + 5)
ax.set_ylim(y_min - 2, y_max + 2)

adjust_text(texts, ax=ax, expand_text=(1.25, 1.35), expand_points=(1.25, 1.35),
            force_text=(0.35, 0.55), force_points=(0.25, 0.45), lim=500,
            arrowprops=dict(arrowstyle="-", color="#cbd5e1", lw=0.45, alpha=0.45))

quadrant_labels = [
    (x_max + 3.5, y_max + 0.5, "High Territory,\nHigh Creation", "right", "top"),
    (x_min - 3.5, y_max + 0.5, "Low Territory,\nHigh Creation", "left", "top"),
    (x_min - 3.5, y_min - 1.0, "Low Territory,\nLow Creation", "left", "bottom"),
    (x_max + 3.5, y_min + 5.0, "High Territory,\nLow Creation", "right", "bottom"),
]
for qx, qy, label, ha, va in quadrant_labels:
    ax.text(qx, qy, label, fontsize=8.3, color="#6b7280", ha=ha, va=va, style="italic",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#f1f5f9", edgecolor="#cbd5e1",
                       linewidth=0.6, alpha=0.85))

ax.set_xlabel("Field Tilt Proxy (%)", fontsize=13)
ax.set_ylabel("Shot Creation Efficiency (%)", fontsize=13)
ax.set_title("Territory Did Not Guarantee Shot Creation", fontsize=16, pad=18)
ax.grid(True, alpha=0.25)

cax = fig.add_axes([0.85, 0.25, 0.022, 0.50])
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Goals per Attempt (%)", fontsize=11)

legend_match_counts = sorted(plot_df["n_matches_total"].unique().astype(int))
legend_handles = []
for mc in legend_match_counts:
    ls = size_min + (mc - min_matches) / (max_matches - min_matches) * (size_max - size_min)
    legend_handles.append(ax.scatter([], [], s=ls, color="#9ca3af", edgecolors="#111827",
                                       linewidth=0.6, alpha=0.65, label=f"{mc} matches"))
size_legend = ax.legend(handles=legend_handles, title="Matches played", loc="lower right",
                          frameon=True, framealpha=0.92, facecolor="#f8fafc", edgecolor="#cbd5e1",
                          fontsize=7.5, title_fontsize=8, labelspacing=0.9)
ax.add_artist(size_legend)

footnote = (
    "Across all 48 teams, territorial control showed no consistent relationship with shot creation efficiency; "
    "point size reflects matches played.\n"
    "Teams with fewer matches may have more volatile tournament averages.\n"
    "Field Tilt Proxy = team final-third entries / both teams' final-third entries × 100; "
    "SCE = attempts at goal / final-third entries × 100.\n"
    f"Colombia and Netherlands had near-identical Field Tilt but opposite profiles: Colombia created more shots per entry but converted fewer ({colombia_gpa:.0f}%);\n"
    f"Netherlands created fewer shots but converted more ({netherlands_gpa:.0f}%).\n"
    "Source: FIFA Match Centre | Full Official Stats only | xG and opponent strength not controlled."
)
fig.text(0.07, 0.035, footnote, fontsize=8.0, color="gray")

output_path = OUTPUT_DIR / "field_tilt_proxy_vs_shot_creation_efficiency_all_completed_matches.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight")
plt.show()

team_metrics_tableau = plot_df.copy().reset_index().rename(columns={"index": "team_name"})
output_csv = OUTPUT_DIR / "team_metrics_field_tilt_tableau_all_completed_matches.csv"
team_metrics_tableau.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"[check] Total teams: {len(plot_df)}")
print(f"[check] Median Field Tilt Proxy: {median_tilt:.2f}%")
print(f"[check] Median SCE: {median_sce:.2f}%")
print(f"[check] Match-count legend items: {legend_match_counts}")
print(f"Saved figure: {output_path}")
print(f"Tableau CSV Saved figure: {output_csv}")

In [ ]:
"""
Optional Analysis:
Group-stage Eliminated Teams Field Tilt Proxy Bar Chart
==========================================================
Purpose:
- Focus on the 16 teams eliminated in the group stage, using team metrics from all completed FIFA Official Stats matches.
- Bar length = Field Tilt.
- Bar color = Shot Creation Efficiency.
- Field Tilt value is shown at the end of each bar.
- Right-side SCE/G/A text labels are removed for cleaner visualization.

Metric notes:
- field_tilt is already expressed as a percentage.
- attempts_per_f3 represents Shot Creation Efficiency (%).
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from pathlib import Path

plt.close("all")


# =========================================================
# 1. Group-stage eliminated teams
# =========================================================

ELIMINATED_TEAMS_RAW = [
    "New Zealand",
    "Qatar",
    "Curacao",
    "Turkiye",
    "Tunisia",
    "Panama",
    "Haiti",
    "Korea Republic",
    "Saudi Arabia",
    "Scotland",
    "Uruguay",
    "Uzbekistan",
    "Czechia",
    "Iraq",
    "Iran",
    "Jordan",
]


TEAM_NAME_ALIASES = {
    "Curacao": ["Curacao", "Curaçao"],
    "Turkiye": ["Turkiye", "Türkiye", "Turkey"],
    "Iran": ["Iran", "IR Iran"],
    "Korea Republic": ["Korea Republic", "South Korea"],
}


def resolve_team_name(name, available_teams):
    candidates = TEAM_NAME_ALIASES.get(name, [name])

    for candidate in candidates:
        if candidate in available_teams:
            return candidate

    return None


available_teams = set(team_metrics.index)

resolved_teams = []
missing_teams = []

for team in ELIMINATED_TEAMS_RAW:
    resolved = resolve_team_name(team, available_teams)

    if resolved is not None:
        resolved_teams.append(resolved)
    else:
        missing_teams.append(team)


if missing_teams:
    raise ValueError(
        "The eliminated-team list contains teams not found in team_metrics: "
        f"{missing_teams}. Fix ELIMINATED_TEAMS_RAW or TEAM_NAME_ALIASES before publishing."
    )

if len(resolved_teams) != 16:
    raise ValueError(f"Expected 16 group-stage eliminated teams, resolved {len(resolved_teams)}: {resolved_teams}")

if len(set(resolved_teams)) != len(resolved_teams):
    duplicated = pd.Series(resolved_teams)[pd.Series(resolved_teams).duplicated()].tolist()
    raise ValueError(f"Duplicate resolved eliminated teams: {duplicated}")

print("[Check] Resolved eliminated teams:")
for team in resolved_teams:
    print(f"  - {team}")


# =========================================================
# 2. Data validation
# =========================================================

required_metrics = [
    "field_tilt",
    "attempts_per_f3"
]

missing_metrics = [
    col for col in required_metrics
    if col not in team_metrics.columns
]

if missing_metrics:
    raise ValueError(
        f"Missing required columns in team_metrics: {missing_metrics}\n"
        f"Current columns: {team_metrics.columns.tolist()}"
    )

if team_metrics.empty:
    raise ValueError("team_metrics is empty.")


# =========================================================
# 3. Filter eliminated teams
# =========================================================

eliminated_metrics = team_metrics.loc[
    team_metrics.index.isin(resolved_teams)
].copy()

if eliminated_metrics.empty:
    raise ValueError("No eliminated teams found in team_metrics.")


DISPLAY_NAME_MAP = {
    "Curaçao": "Curacao",
    "Türkiye": "Turkiye",
    "Turkey": "Turkiye",
    "IR Iran": "Iran",
}


eliminated_metrics["display_team_name"] = [
    DISPLAY_NAME_MAP.get(team, team)
    for team in eliminated_metrics.index
]


# Sort by Field Tilt in ascending order
eliminated_metrics = eliminated_metrics.sort_values(
    "field_tilt",
    ascending=True
)


# =========================================================
# 4. Color setting
# =========================================================

norm = mcolors.Normalize(
    vmin=eliminated_metrics["attempts_per_f3"].min(),
    vmax=eliminated_metrics["attempts_per_f3"].max()
)

cmap = plt.get_cmap("YlOrRd")

bar_colors = [
    cmap(norm(value))
    for value in eliminated_metrics["attempts_per_f3"]
]


# =========================================================
# 5. Figure setup
# =========================================================

fig, ax = plt.subplots(figsize=(13, 8.5))

fig.patch.set_facecolor("#0f172a")
ax.set_facecolor("#111827")

team_names = eliminated_metrics["display_team_name"]
field_tilt_values = eliminated_metrics["field_tilt"]

y_pos = np.arange(len(eliminated_metrics))


# =========================================================
# 6. Background track bars
# =========================================================

ax.barh(
    y_pos,
    [100] * len(eliminated_metrics),
    color="#1f2937",
    edgecolor="none",
    height=0.72,
    alpha=0.75,
    zorder=1
)


# =========================================================
# 7. Main bars
# =========================================================

bars = ax.barh(
    y_pos,
    field_tilt_values,
    color=bar_colors,
    edgecolor="#f9fafb",
    linewidth=0.8,
    height=0.72,
    zorder=3
)


# =========================================================
# 8. 50% reference line
# =========================================================

ax.axvline(
    50,
    linestyle="--",
    color="#e5e7eb",
    linewidth=1.2,
    alpha=0.9,
    zorder=2
)

ax.text(
    50,
    len(eliminated_metrics) - 0.2,
    "50% Field Tilt Proxy",
    color="#e5e7eb",
    fontsize=8,
    ha="center",
    va="bottom"
)


# =========================================================
# 9. Field Tilt value labels only
# =========================================================

for i, (_, row) in enumerate(eliminated_metrics.iterrows()):
    field_tilt = row["field_tilt"]

    ax.text(
        field_tilt + 1.0,
        i,
        f"{field_tilt:.1f}%",
        va="center",
        ha="left",
        fontsize=9,
        color="#f9fafb",
        fontweight="bold",
        zorder=5
    )


# =========================================================
# 10. Axes and labels
# =========================================================

ax.set_yticks(y_pos)

ax.set_yticklabels(
    team_names,
    fontsize=10,
    color="#f9fafb"
)

ax.set_xlim(0, 100)

ax.set_xlabel(
    "Field Tilt Proxy (%)",
    fontsize=12,
    color="#f9fafb",
    labelpad=10
)

ax.set_ylabel(
    "Group-stage eliminated team",
    fontsize=12,
    color="#f9fafb",
    labelpad=10
)

ax.set_title(
    "Territorial Control among Group-stage Eliminated Teams",
    fontsize=17,
    color="#f9fafb",
    pad=22,
    fontweight="bold"
)

ax.grid(
    axis="x",
    color="#374151",
    alpha=0.5,
    linewidth=0.8,
    zorder=0
)

ax.tick_params(axis="x", colors="#d1d5db")
ax.tick_params(axis="y", colors="#f9fafb")

for spine in ax.spines.values():
    spine.set_visible(False)


# =========================================================
# 11. Colorbar
# =========================================================

sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    shrink=0.75,
    pad=0.03
)

cbar.set_label(
    "Shot Creation Efficiency (%)",
    fontsize=10,
    color="#f9fafb",
    labelpad=10
)

cbar.ax.yaxis.set_tick_params(color="#d1d5db")
plt.setp(cbar.ax.get_yticklabels(), color="#d1d5db", fontsize=9)
cbar.outline.set_edgecolor("#4b5563")


# =========================================================
# 12. Footnote
# =========================================================

footnote = (
    "Bar length represents Field Tilt. Bar color represents Shot Creation Efficiency.\n"
    "Field Tilt Proxy = team final-third entries / both teams' final-third entries × 100.\n"
    "Shot Creation Efficiency = attempts at goal / final-third entries × 100. "
    "xG not included; opponent strength not controlled."
)

fig.text(
    0.08,
    0.025,
    footnote,
    fontsize=8.5,
    color="#9ca3af"
)


# =========================================================
# 13. Layout and save
# =========================================================

plt.tight_layout(rect=[0, 0.08, 1, 1])

output_path = OUTPUT_DIR / "group_stage_eliminated_teams_field_tilt_proxy_bar_all_completed_metrics.png"

fig.savefig(
    output_path,
    dpi=240,
    bbox_inches="tight",
    facecolor=fig.get_facecolor()
)

plt.show()


# =========================================================
# 14. Save Tableau-ready CSV
# =========================================================

eliminated_tableau = eliminated_metrics.copy().reset_index()
eliminated_tableau = eliminated_tableau.rename(columns={"index": "team_name"})
eliminated_tableau["team_status"] = "Eliminated"

output_csv = OUTPUT_DIR / "group_stage_eliminated_teams_field_tilt_tableau_all_completed_metrics.csv"

eliminated_tableau.to_csv(
    output_csv,
    index=False,
    encoding="utf-8-sig"
)


# =========================================================
# 15. Validation output
# =========================================================

print(f"\n[Check] Eliminated teams plotted: {len(eliminated_metrics)}")
print(f"[Check] Field Tilt range: {eliminated_metrics['field_tilt'].min():.2f}% ~ {eliminated_metrics['field_tilt'].max():.2f}%")
print(f"[Check] Shot Creation Efficiency range: {eliminated_metrics['attempts_per_f3'].min():.2f}% ~ {eliminated_metrics['attempts_per_f3'].max():.2f}%")
print(f"Saved figure: {output_path}")
print(f"Saved Tableau CSV: {output_csv}")





In [ ]:
"""
Optional Step: Group-stage Eliminated Teams Diagnostic Table
====================================================
Purpose:
- Classify group-stage eliminated teams using the all-completed-match team metrics.
- Identify whether each team struggled with territory, shot creation, or both.
- Save a Tableau-ready diagnostic CSV.
"""

import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# 1. Check required dataframe
# =========================================================

if "eliminated_metrics" not in globals():
    raise ValueError(
        "eliminated_metrics does not exist. "
        "Run the eliminated teams bar chart code first."
    )

required_cols = [
    "field_tilt",
    "attempts_per_f3",
    "goals_per_attempt",
    "display_team_name"
]

missing_cols = [
    col for col in required_cols
    if col not in eliminated_metrics.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}\n"
        f"Current columns: {eliminated_metrics.columns.tolist()}"
    )


# =========================================================
# 2. Thresholds
# =========================================================
# Field Tilt Proxy: 50% = territorial balance line
# Shot Creation Efficiency: median among eliminated teams

FIELD_TILT_THRESHOLD = 50
SCE_THRESHOLD = eliminated_metrics["attempts_per_f3"].median()

print(f"[Check] Field Tilt threshold: {FIELD_TILT_THRESHOLD:.1f}%")
print(f"[Check] Shot Creation Efficiency threshold: {SCE_THRESHOLD:.2f}%")


# =========================================================
# 3. Classification function
# =========================================================

def classify_eliminated_team(row):
    field_tilt = row["field_tilt"]
    sce = row["attempts_per_f3"]

    if field_tilt >= FIELD_TILT_THRESHOLD and sce >= SCE_THRESHOLD:
        return "Territory gained, efficient shot creation"

    elif field_tilt >= FIELD_TILT_THRESHOLD and sce < SCE_THRESHOLD:
        return "Territory gained, limited shot creation"

    elif field_tilt < FIELD_TILT_THRESHOLD and sce >= SCE_THRESHOLD:
        return "Limited territory, efficient shot creation"

    else:
        return "Limited territory, limited shot creation"


def write_interpretation(row):
    team = row["display_team_name"]
    category = row["elimination_profile"]

    if category == "Territory gained, efficient shot creation":
        return (
            f"{team} reached advanced areas relatively often and converted those entries "
            f"into shots at an above-median rate, suggesting that elimination may be linked "
            f"more to finishing, defensive issues, or match context than territorial control."
        )

    elif category == "Territory gained, limited shot creation":
        return (
            f"{team} recorded above-50% Field Tilt but below-median Shot Creation Efficiency, "
            f"suggesting that territorial advantage did not translate efficiently into attempts at goal."
        )

    elif category == "Limited territory, efficient shot creation":
        return (
            f"{team} had below-50% Field Tilt but above-median Shot Creation Efficiency, "
            f"suggesting fewer advanced entries but relatively effective shot generation when entering the final third."
        )

    else:
        return (
            f"{team} recorded below-50% Field Tilt and below-median Shot Creation Efficiency, "
            f"suggesting difficulties both in reaching advanced areas and converting entries into shots."
        )


# =========================================================
# 4. Create diagnostic table
# =========================================================

eliminated_diagnostic = eliminated_metrics.copy()

eliminated_diagnostic["elimination_profile"] = eliminated_diagnostic.apply(
    classify_eliminated_team,
    axis=1
)

eliminated_diagnostic["interpretation"] = eliminated_diagnostic.apply(
    write_interpretation,
    axis=1
)

eliminated_diagnostic["field_tilt_rank_among_eliminated"] = (
    eliminated_diagnostic["field_tilt"]
    .rank(ascending=False, method="min")
    .astype(int)
)

eliminated_diagnostic["shot_creation_efficiency_rank_among_eliminated"] = (
    eliminated_diagnostic["attempts_per_f3"]
    .rank(ascending=False, method="min")
    .astype(int)
)

eliminated_diagnostic["goals_per_attempt_rank_among_eliminated"] = (
    eliminated_diagnostic["goals_per_attempt"]
    .rank(ascending=False, method="min")
    .astype(int)
)


# =========================================================
# 5. Select final columns
# =========================================================

final_cols = [
    "display_team_name",
    "field_tilt",
    "attempts_per_f3",
    "goals_per_attempt",
    "field_tilt_rank_among_eliminated",
    "shot_creation_efficiency_rank_among_eliminated",
    "goals_per_attempt_rank_among_eliminated",
    "elimination_profile",
    "interpretation"
]

eliminated_diagnostic_table = eliminated_diagnostic[final_cols].copy()

eliminated_diagnostic_table = eliminated_diagnostic_table.sort_values(
    ["elimination_profile", "field_tilt"],
    ascending=[True, False]
)

# Round values for display
round_cols = [
    "field_tilt",
    "attempts_per_f3",
    "goals_per_attempt"
]

eliminated_diagnostic_table[round_cols] = (
    eliminated_diagnostic_table[round_cols].round(2)
)


# =========================================================
# 6. Display table
# =========================================================

display(eliminated_diagnostic_table)


# =========================================================
# 7. Profile count summary
# =========================================================

profile_summary = (
    eliminated_diagnostic_table
    .groupby("elimination_profile", as_index=False)
    .agg(
        n_teams=("display_team_name", "count"),
        avg_field_tilt=("field_tilt", "mean"),
        avg_shot_creation_efficiency=("attempts_per_f3", "mean"),
        avg_goals_per_attempt=("goals_per_attempt", "mean")
    )
)

profile_summary[[
    "avg_field_tilt",
    "avg_shot_creation_efficiency",
    "avg_goals_per_attempt"
]] = profile_summary[[
    "avg_field_tilt",
    "avg_shot_creation_efficiency",
    "avg_goals_per_attempt"
]].round(2)

display(profile_summary)


# =========================================================
# 8. Save outputs
# =========================================================

output_diagnostic_csv = OUTPUT_DIR / "group_stage_eliminated_teams_diagnostic_table_all_completed_metrics.csv"
output_profile_summary_csv = OUTPUT_DIR / "group_stage_eliminated_teams_profile_summary_all_completed_metrics.csv"

eliminated_diagnostic_table.to_csv(
    output_diagnostic_csv,
    index=False,
    encoding="utf-8-sig"
)

profile_summary.to_csv(
    output_profile_summary_csv,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved diagnostic table: {output_diagnostic_csv}")
print(f"Saved profile summary: {output_profile_summary_csv}")

